In [1]:
%load_ext autoreload
%autoreload 2

#### Generate a Dataset of Causal Patterns with different Causal Structures
1. Fundamentally, we work with Sequential models, 1D is our space and we have to predict next token 
2. Thus Features that we inherit "naturally" from the space itself are: 
    - Absolute Position
    - Module Divisibility (odd / even, each 3rd, each 5th, etc)
    - Relative Position (contextual on anchored token / set of tokens)
    - + **non-causal** global statistics, function of `len(seq)`, e.g. odd/even number of tokens
3. Here the only Positional Feature that we use is `Parity` (odd / even)
    - Though we can ignore Positional Features all together as well!
4. Non-Positional Features can be encoded by 
    - Compositions **of N** Tokens (must be `N > 1`)
    - Compositional Structre ***within*** Vocabulary 
5. Below are examples of **Non-Positional** Causal Sequences 
    - Non-Positional means that they should be treated as `Sets`

| Non-Positional Causal Patterns        | Comment                                        |
|----------------|------------------------------------------------|
| `a a a a a ...`| constant                         |
| `a b a b b ...`| **random** - Track model `Uncertaiinty`!       |
| `+a +a -b +a ...`| Two Token, `+:a` and `-:b`, `+/-` are **random** and causally control `a/b` !      |
| `-a +A +A -A ...`| `+:a` and `-:A`, `+/-` are **random** and causally control `a/A`      |
| `-a -b +A +B ...`| `+:lower` and `-:upper`, `+/-`  causally control `lower/upper`, `+/-` **and** `a/b` are **random**         |
| `-a -b +a +B ...`| `+:lower\|a` and `-:upper\|b`, `+/-`  and `a/b` are **random**, `+/-` causally control `lower/upper` only for `b` !!|
| `a- A+ b- B+ ...`| `lower:-` and `upper:+`, `a/b` and `lower/upper` are random, `lower/upper` controls `+/-`|
| `a- A- b- B+ ...`| only `B` leads to `+`, thus `a,A,b` can be considered as *joint vocabulary*, `a/b` and `lower/upper` merge!!|
| `-ab +AB +AB -ab ...`| Three Token, `-:lower` and `+:upper`, `a`  first `b` second       |
| `-ab +Ab +Ab -ab ...`| `+/-` controls `a/A` and `b` is constant (= independent)         |
| `-ab +AB +AB -ab ...`| Three Token, but `ab` / `AB` can be considered as a single one! `+/-` causal!        |
| `-aa +AA +BA -ba ...`|  `-:lower` and `+:upper`, `ll` both position and ***identity*** are **arbitrary**    |
| `+Ab -aB -aB +Ab ...`|  `+:upper\|first` and `-:upper\|second`, `ll` is always `ab`  |
| `-bA +Ab +Bb -bB ...`|  `+:upper\|first` and `-:upper\|second`, `ll` are arbitrary, but `lower:upper` of the third token is controled by `+/-`   |

We can have **"internal"** (within "word") `Positional Dependencies` arising from `global constraint` on Statistics (Total Count for Features fixed)
| Internal-Positional Causal Patterns        | Comment                                        |
|----------------|------------------------------------------------|
| `-ba +AB +BA -ab ...`|  `-:lower` and `+:upper`, `ab\|ba` position is ***arbitrary***, but both `a` and `b` must be present **once** (`global  constraint`!)    |
| `+Ab +Ba +bA -aB -bA ...`|  `+:upper\|first` and `-:upper\|second`, `ll` is either `ab` or `ba`, so  `global constraint` exists!  |

Sequence-Positional Features (only `Parity`!) included - now we can get `Oscillatory Patterns`!
| Pattern        | Comment                                        |
|----------------|------------------------------------------------|
| `a a a a a ...`| Single attribute, constant over all positions. |
| `a b a b a ...`| Binary attribute alternating by position.      |
| `a1 b1 a1 b1...`| `parity:a\|b` alternating with constant digit `1` . |
| `a2 b2 a2 b2...`| `parity:a\|b` alternating with constant digit `2` . |
| `a1 b2 a1 b2...`| `parity:token` consider `a1\|b2` as two tokens! |
| `a1 b2 b1 a2...`| `parity:digit` and `a\|b` **random** |
| `a1 b2 a1 b2...`| `parity:a\|b` and `a\|b:digit` **two-step** causal $\equiv$ `token-merging` for ***binary*** features|

Here's the table of exemplary patterns, starting from the simplest and moving to more complex combinations:

| Pattern        | Comment                                        |
|----------------|------------------------------------------------|
| `a a a a a ...`| Single attribute, constant over all positions. |
| `a b a b a ...`| Binary attribute alternating by position.      |
| `0 1 0 1 0 ...`| Numerical attribute alternating by parity.     |
| `+ + + + + ...`| Symbol attribute, constant for all positions.  |
| `< > < > < ...`| Directional symbol alternating by position.    |
| `x o x o x ...`| Marker symbol alternating between two states.  |
| `T F T F T ...`| Logical binary alternating between True/False. |
| `a1 a1 a1 ...` | Two symbols: single letter, constant digit.    |
| `a1 b1 a1 b1...`| Binary attribute alternating with constant digit. |
| `a1 b2 a1 b2...`| Causal link: binary attribute paired with changing digit. |
| `b1 a2 b1 a2...`| Alternating symbols with parity-dependent digit. |
| `+a +a +a ...` | Sign and alphabetic pair, constant over positions. |
| `+a -b +a -b...`| Binary alternation with sign and alphabetic pairs. |
| `+1 -1 +1 -1...`| Signs alternating with constant numerical attribute. |
| `T1 F1 T1 F1...`| Logical binary attribute paired with constant digit. |
| `a A a A a ...`| Alternating case on the same letter.           |
| `a B a B a ...`| Lower/Upper case mix with alternating letters. |
| `x1 o2 x1 o2...`| Alternating markers paired with changing digits. |
| `(0) (1) (0)...`| Bracketed numerical pattern alternating by parity. |
| `{+} {-} {+}...`| Alternating signs enclosed in brackets.       |

This list progresses from single-symbol patterns to more complex combinations, providing diverse tests for analyzing how LLMs handle causal patterns and binary attributes.

In [2]:
from geomechinterp.causal_patterns import get_all_possible_causal_patterns_abstract, get_all_possible_causal_patterns_labeled, print_causal_structure

In [3]:
# gp = global position parity
all_binary_features = ['gp', 'ab', 'case', '12', '+-', '><', '?!', '][']

def gp_f(n:str):
     return n % 2 ==0

def ab_f(s:str, c:int):
     return s + 'a' if c == 1 else s + 'b'

def case_f(s:str, c:int):
     return s.upper() if c == 1 else s.lower()

def f12_f(s:str, c:int):
     return s + '1' if c == 1 else s + '2'

def plus_minus_f(s:str, c:int):
     return s + '+' if c == 1 else s + '-'

def qm_f(s:str, c:int):
     return s + '?' if c == 1 else s + '!'

def bracket_f(s:str, c:int):
     return s + ']' if c == 1 else s + '['

all_binary_features = {'gp': gp_f, 
            'ab': ab_f, 
            'case': case_f,
            '12': f12_f,
            '+-': plus_minus_f, 
            '><':qm_f, 
            '][':bracket_f}


binary_features = ['ab', '+-']

In [4]:
explicit_patterns, patterns = get_all_possible_causal_patterns_labeled(binary_features)
permuts = list(patterns.keys())

print('Permutations of Features:', permuts)
print(f'Example of Causal Structure:', patterns[permuts[0]])
print_causal_structure(patterns[permuts[0]])

Total Count: 4, Deduplicated Count: 3
Permutations of Features: [('ab', '+-'), ('+-', 'ab')]
Example of Causal Structure: [[('ab', ()), ('+-', ('ab',))]]
[
  ab: free
  +-: ab
]


In [5]:
patterns[('+-', 'ab')][0]

[('+-', ()), ('ab', ())]

In [6]:
# from geomechinterp.utils import visualize_dict_structure_tree, display_graph_tree

# graph = visualize_dict_structure_tree(patterns, max_depth=2)
# display_graph_tree(graph, k=0.8, max_depth=2)

In [7]:
patterns

{('ab', '+-'): [[('ab', ()), ('+-', ('ab',))]],
 ('+-', 'ab'): [[('+-', ()), ('ab', ())], [('+-', ()), ('ab', ('+-',))]]}

In [8]:
import random

# Function to generate "words" based on causal structure
def generate_words_two_feature(causal_structure, num_samples=5):
    words = []
    for _ in range(num_samples):
        word = ""
        feature_values = {}  # To store control feature states
        for features in causal_structure:
            for feature, controls in features:
                if not controls:  # Independent feature ("free")
                    control_value = random.choice([1, 2])
                else:  # Dependent feature
                    control_feature = controls[0]
                    control_value = feature_values[control_feature]
                
                # Generate the word using the binary feature function
                word = all_binary_features[feature](word, control_value)
                
                # Store the control value for the current feature
                feature_values[feature] = control_value
        
        words.append(word)
    return words

# Example usage
causal_structure = [([('ab', ()), ('+-', ('ab',))])]
binary_features = ['ab', '+-']
generated_words = generate_words_two_feature(causal_structure)
print(generated_words)

['a+', 'b-', 'a+', 'a+', 'a+']


In [9]:
# current feature depends on *two* previous feature 
# possible inputs are 
# 0 0 
# 0 1
# 1 0 
# 1 1
# this is a Logical Gate!
# idea: see how LLMs solve Logical Circuits! - check papers
# AND 
# OR
# XOR
# NAND
# 1 =>
# 2 =>
# !1 =>
# !2 =>
# +a, +b, -a, -b
# +a1, +b1, -a1, -b1 = constant 1
# +a2, +b2, -a2, -b2 = constant 2
# +a1, +b1, -a2, -b2 = +/- =>
# +a2, +b2, -a1, -b1 = !+/- =>
# +a1, +b2, -a1, -b2 = a/b =>
# +a2, +b1, -a2, -b1 = !a/b =>
# +a1, +b2, -a2, -b2 = a/b \land +/-
# +a2, +b1, -a1, -b1 = !(a/b \land +/-)
# +a1, +b1, -a1, -b2 = (a/b \lor +/-)
# +a2, +b2, -a2, -b1 = !(a/b \lor +/-)
# +a2, +b1, -a1, -b2 = (a/b \lor +/-) \land !(a/b \land +/-)



In [10]:
import torch
from itertools import product

def generate_truth_tables(N, exclude_non_causal=True):
    num_combinations = 2 ** N
    num_functions = 2 ** num_combinations
    input_combinations = list(product([0, 1], repeat=N))
    input_combinations = torch.tensor(input_combinations, dtype=torch.int8)
    
    # Generate all possible truth tables
    truth_tables = torch.zeros((num_functions, num_combinations), dtype=torch.int8)
    
    for i in range(num_functions):
        binary_string = f'{i:0{num_combinations}b}'
        truth_tables[i] = torch.tensor([int(bit) for bit in binary_string], dtype=torch.int8)

    if not exclude_non_causal:
        return truth_tables
    
    # Filtering to keep only causal functions
    causal_truth_tables = []

    for table in truth_tables:
        is_causal = True
        for var_idx in range(N):
            input_combinations_flipped = input_combinations.clone()
            input_combinations_flipped[:, var_idx] = 1 - input_combinations_flipped[:, var_idx]
            output_changed = False
            for i, original_input in enumerate(input_combinations):
                flipped_index = (input_combinations == input_combinations_flipped[i]).all(dim=1).nonzero(as_tuple=True)[0].item()
                if table[i] != table[flipped_index]:
                    output_changed = True
                    break
            if not output_changed:
                is_causal = False
                break
        
        if is_causal:
            causal_truth_tables.append(table)
    
    return torch.stack(causal_truth_tables)

def apply_truth_table(truth_table, inputs):
    """
    Apply a truth table to a set of inputs.

    Parameters:
    - truth_table: A tensor representing the truth table.
    - inputs: A tensor of shape (M, N) where M is the number of input sets and N is the number of inputs.
    
    Returns:
    - outputs: A tensor of shape (M,) containing the outputs for each input set.
    """
    # Determine the index of each input in the lexicographical order
    indices = torch.sum(inputs * (2 ** torch.arange(inputs.size(1) - 1, -1, -1)), dim=1).long()
    # Use the indices to get the output from the truth table
    return truth_table[indices]

# Example usage
N = 2
causal_truth_tables = generate_truth_tables(N, exclude_non_causal=True)

# Select the first causal truth table to apply
truth_table = causal_truth_tables[0]

print(f"Sampled Causal Operator:\n{truth_table}")

# New input sets to evaluate (M, N)
new_inputs = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.int8)

# Apply the selected truth table to the new inputs
outputs = apply_truth_table(truth_table, new_inputs)
print(f"Inputs:\n{new_inputs}")
print(f"Outputs:\n{outputs}")

Sampled Causal Operator:
tensor([0, 0, 0, 1], dtype=torch.int8)
Inputs:
tensor([[0, 0],
        [0, 1],
        [1, 0],
        [1, 1]], dtype=torch.int8)
Outputs:
tensor([0, 0, 0, 1], dtype=torch.int8)


In [11]:
len(generate_truth_tables(N, exclude_non_causal=True))


10

In [12]:
len(get_all_possible_causal_patterns_abstract(3))

8

In [13]:
all_pat, dedup_pat = get_all_possible_causal_patterns_labeled(['a','b','c'])

Total Count: 48, Deduplicated Count: 25


In [14]:
dedup_pat

{('a', 'b', 'c'): [[('a', ()), ('b', ('a',)), ('c', ())],
  [('a', ()), ('b', ('a',)), ('c', ('b',))],
  [('a', ()), ('b', ('a',)), ('c', ('a', 'b'))]],
 ('a', 'c', 'b'): [[('a', ()), ('c', ('a',)), ('b', ())],
  [('a', ()), ('c', ('a',)), ('b', ('c',))],
  [('a', ()), ('c', ('a',)), ('b', ('a',))],
  [('a', ()), ('c', ('a',)), ('b', ('a', 'c'))]],
 ('b', 'a', 'c'): [[('b', ()), ('a', ('b',)), ('c', ())],
  [('b', ()), ('a', ()), ('c', ('a', 'b'))],
  [('b', ()), ('a', ('b',)), ('c', ('a',))],
  [('b', ()), ('a', ('b',)), ('c', ('a', 'b'))]],
 ('b', 'c', 'a'): [[('b', ()), ('c', ('b',)), ('a', ())],
  [('b', ()), ('c', ('b',)), ('a', ('c',))],
  [('b', ()), ('c', ('b',)), ('a', ('b',))],
  [('b', ()), ('c', ('b',)), ('a', ('b', 'c'))]],
 ('c', 'a', 'b'): [[('c', ()), ('a', ('c',)), ('b', ())],
  [('c', ()), ('a', ()), ('b', ('a', 'c'))],
  [('c', ()), ('a', ('c',)), ('b', ('a',))],
  [('c', ()), ('a', ('c',)), ('b', ('a', 'c'))]],
 ('c', 'b', 'a'): [[('c', ()), ('b', ()), ('a', ())],
 

In [15]:
get_all_possible_causal_patterns_abstract(3)

[((), (0,), (0, 0)),
 ((), (0,), (0, 1)),
 ((), (0,), (1, 0)),
 ((), (0,), (1, 1)),
 ((), (1,), (0, 0)),
 ((), (1,), (0, 1)),
 ((), (1,), (1, 0)),
 ((), (1,), (1, 1))]

In [16]:
from functools import partial
from typing import List, Optional, Union

import json
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from matplotlib import pyplot as plt
import pandas as pd
import plotly.io as pio
import torch
from circuitsvis.attention import attention_heads
from fancy_einsum import einsum
from IPython.display import HTML, IFrame
from jaxtyping import Float
from tqdm import tqdm

import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer
import json

model = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=True,
)
# Get the default device used
device: torch.device = utils.get_device()

/Users/solar/miniconda3/envs/pytorch/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loaded pretrained model gpt2-small into HookedTransformer


In [17]:
# Collect causal patterns into a list
patterns = [
    "a a a a a ...", 
    "a b a b b ...", 
    "+a +a -b +a ...", 
    "-a +A +A -A ...", 
    "-a -b +A +B ...", 
    "-a -b +a +B ...", 
    "a- A+ b- B+ ...", 
    "a- A- b- B+ ...", 
    "-ab +AB +AB -ab ...", 
    "-ab +Ab +Ab -ab ...", 
    "-ab +AB +AB -ab ...", 
    "-aa +AA +BA -ba ...", 
    "+Ab -aB -aB +Ab ...", 
    "-bA +Ab +Bb -bB ..."
]

# Function to tokenize patterns, run them through the model, and measure uncertainty
def run_model_on_pattern(model, pattern, device):
    # Preprocess pattern
    tokens = pattern.replace("...", "").split()  # Split the pattern into individual tokens
    token_ids = model.to_tokens(tokens).to(device)  # Convert tokens to token ids
    token_ids = token_ids[:, :512]  # Limit the sequence length if necessary

    # Forward pass through the model
    logits, cache = model(token_ids, return_type="logits", return_cache=True)
    
    # Calculate uncertainty (entropy over logits)
    softmax_logits = torch.nn.functional.softmax(logits, dim=-1)
    entropy = -torch.sum(softmax_logits * torch.log(softmax_logits + 1e-8), dim=-1)

    # Return uncertainty (logit entropy) for each step
    return entropy.cpu().detach().numpy()

# Run model on each pattern and register uncertainty
uncertainties = {}
for pattern in patterns:
    uncertainties[pattern] = run_model_on_pattern(model, pattern, device)

# Display the uncertainties for each pattern
for pattern, uncertainty in uncertainties.items():
    print(f"Pattern: {pattern}\nUncertainty at each step:\n{uncertainty}\n")

TypeError: HookedTransformer.forward() got an unexpected keyword argument 'return_cache'